In [1]:
import os
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE and os.path.exists('/content')

print(f"Ortam: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Yerel'}")

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive bağlandı.")

Ortam: Colab
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive bağlandı.


In [2]:
import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Tuple

import sys, shutil, subprocess


In [3]:

SUPER_CLASSES: List[str] = [
    "acute_cholecystitis",        # 0 → label 1
    "kidney_ureter_stone",        # 1 → label 2
    "acute_pancreatitis",         # 2 → label 3
    "aortic_aneurysm_dissection", # 3 → label 4
    "acute_appendicitis",         # 4 → label 5
    "acute_diverticulitis",       # 5 → label 6
]


In [4]:
import os, sys, shutil, subprocess, json
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List

# ── Ortam bazlı yol konfigürasyonu ────────────────────────────────────────
if IS_KAGGLE:
    KAGGLE_INPUT = Path('/kaggle/input/datasets/ramazan2020/abdomen-acikveri')
    DATA_ROOT    = KAGGLE_INPUT
    EGITIM_DIR   = DATA_ROOT / "Egitim Verisi"
    SPLIT_DIR    = DATA_ROOT / "outputs/splits"
    CODE         = DATA_ROOT
    DET_DATA_DIR = DATA_ROOT / "det_data"
    NND_ROOT     = Path('/kaggle/working/nndet')
    NIFTI_DIR    = NND_ROOT / "nifti"

elif IS_COLAB:
    DRIVE_BASE   = Path('/content/drive/MyDrive/Abdomen')
    DATA_ROOT    = DRIVE_BASE
    EGITIM_DIR   = DATA_ROOT / "Egitim Verisi"
    SPLIT_DIR    = DRIVE_BASE / "outputs/splits"
    CODE         = DRIVE_BASE
    DET_DATA_DIR = DRIVE_BASE / "outputs/det_data"
    NND_ROOT     = DRIVE_BASE / "nndet"
    NIFTI_DIR    = Path('/content/nndet/nifti')   # Colab yerel disk (hızlı I/O)

else:
    from dotenv import load_dotenv
    load_dotenv()
    DATA_ROOT    = Path(os.environ.get("TR_ABDOMEN_BASE", "."))
    PROJE        = Path(os.environ.get("TR_ABDOMEN_PROJE", "."))
    CODE         = Path(os.environ.get("TR_ABDOMEN_CODE",  "."))
    EGITIM_DIR   = DATA_ROOT / "Egitim Verisi"
    SPLIT_DIR    = PROJE / "outputs/splits"
    DET_DATA_DIR = PROJE / "outputs/det_data"
    NND_ROOT     = PROJE / "outputs/nndet"
    NIFTI_DIR    = NND_ROOT / "nifti"

if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))


### nnU-Net v2 Kurulumu

nnDetection yerine **nnU-Net v2** kullanıyoruz:
- `pip install nnunetv2` → bitti, CUDA derleme yok
- Semantic segmentation ile eğitim → predicted mask'lerden connected-component bbox çıkarımı
- Python 3.12 / PyTorch 2.x tam uyumlu

In [ ]:
!pip install nnunetv2 scipy


In [ ]:
import importlib

print("nnU-Net v2 + scipy kuruluyor...")
_r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "nnunetv2", "scipy", "-q"],
    capture_output=True, text=True
)
if _r.returncode != 0:
    print("✗ Kurulum hatası:"); print(_r.stderr[-800:])
else:
    importlib.invalidate_caches()
    try:
        import nnunetv2
    except ImportError as e:
        print(f"✗ import hatası: {e}")

# Komut yollarını doğrula
import shutil as _sh, sysconfig as _sc
for _cmd in ["nnUNetv2_plan_and_preprocess", "nnUNetv2_train", "nnUNetv2_predict"]:
    _p = _sh.which(_cmd) or str(Path(_sc.get_path("scripts")) / _cmd)
    print(f"  {_cmd}: {_p}  (var={Path(_p).exists()})")


In [5]:

# ── nnU-Net v2 yolları ─────────────────────────────────────────────────────
# Dataset dizini: {nnUNet_raw}/Dataset100_Abdomen/imagesTr|labelsTr
NNUNET_RAW          = NND_ROOT / "nnunet_raw"
NNUNET_PREPROCESSED = NND_ROOT / "nnunet_preprocessed"
NNUNET_RESULTS      = NND_ROOT / "nnunet_results"

DATASET_ID   = 100
DATASET_NAME = "Abdomen"
DATASET_DIR  = NNUNET_RAW / f"Dataset{DATASET_ID}_{DATASET_NAME}"

os.environ["nnUNet_raw"]          = str(NNUNET_RAW)
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
os.environ["nnUNet_results"]      = str(NNUNET_RESULTS)
os.environ["OMP_NUM_THREADS"]     = "1"

for p in [NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS,
          DATASET_DIR / "imagesTr", DATASET_DIR / "labelsTr", NIFTI_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def load_fold(fold: int, split: str) -> List[int]:
    p = SPLIT_DIR / f"fold{fold}_{split}.csv"
    return pd.read_csv(p)["Case Number"].astype(int).tolist()

fold = 0
print(f"nnUNet_raw         : {os.environ['nnUNet_raw']}")
print(f"nnUNet_preprocessed: {os.environ['nnUNet_preprocessed']}")
print(f"nnUNet_results     : {os.environ['nnUNet_results']}")
print(f"DATASET_DIR        : {DATASET_DIR}")
print(f"NIFTI_DIR          : {NIFTI_DIR}")
print(f"EGITIM_DIR         : {EGITIM_DIR}  → var? {EGITIM_DIR.exists()}")


nnUNet_raw         : /content/drive/MyDrive/Abdomen/nndet/nnunet_raw
nnUNet_preprocessed: /content/drive/MyDrive/Abdomen/nndet/nnunet_preprocessed
nnUNet_results     : /content/drive/MyDrive/Abdomen/nndet/nnunet_results
DATASET_DIR        : /content/drive/MyDrive/Abdomen/nndet/nnunet_raw/Dataset100_Abdomen
NIFTI_DIR          : /content/nndet/nifti
EGITIM_DIR         : /content/drive/MyDrive/Abdomen/Egitim Verisi  → var? True


In [8]:
# ── Preprocessed veriyi Drive'dan Colab local diske kopyala ───────────────
# Google Drive FUSE üzerinden çok-thread'li NPZ okuma bağlantıyı koparıyor.
# Çözüm: eğitim başlamadan önce preprocessed klasörü /content/ altına al.
import shutil, time
IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE and os.path.exists('/content')

LOCAL_PREPROCESSED = Path('/content/nnunet_preprocessed')
LOCAL_RESULT = Path('/content/nnUNet_results')

In [ ]:


if IS_COLAB:
    drive_pre   = NNUNET_PREPROCESSED
    dataset_sub = f'Dataset{DATASET_ID}_{DATASET_NAME}'
    local_sub   = LOCAL_PREPROCESSED / dataset_sub
    if not local_sub.exists():
        src = drive_pre / dataset_sub
        if not src.exists():
            raise FileNotFoundError(
                f'Preprocessed klasörü bulunamadı: {src}\n'
                'Önce plan_and_preprocess hücresini çalıştırın.'
            )
        print(f'Preprocessed verisi kopyalanıyor: {src}')
        print(f'  → {local_sub}')
        t0 = time.time()
        shutil.copytree(str(src), str(local_sub))
        print(f'  ✓ Tamamlandı ({time.time()-t0:.0f}s)')
    else:
        print(f'Local preprocessed zaten mevcut: {local_sub}')


else:
    print('Colab dışı ortam — kopyalama atlandı.')


In [9]:
# Ortam değişkenini yerel yola güncelle
os.environ['nnUNet_preprocessed'] = str(LOCAL_PREPROCESSED)
os.environ['nnUNet_results'] = str(LOCAL_RESULT)
NNUNET_PREPROCESSED = LOCAL_PREPROCESSED
print(f"nnUNet_preprocessed: {os.environ['nnUNet_preprocessed']}")
print(f"nnUNet_results     : {os.environ['nnUNet_results']}")


nnUNet_preprocessed: /content/nnunet_preprocessed
nnUNet_results     : /content/nnUNet_results


In [10]:

import sysconfig as _sc
def _nnunet_cmd(name: str) -> str:
    p = shutil.which(name)
    if p: return p
    for d in [_sc.get_path("scripts"), str(Path(sys.executable).parent),
               "/usr/local/bin", "/opt/conda/bin"]:
        c = Path(d) / name
        if c.exists(): return str(c)
    return name


_cmd_train = _nnunet_cmd("nnUNetv2_train")
print(f"nnUNetv2_train     : {_cmd_train}  (var={Path(_cmd_train).exists()})")

nnUNetv2_train     : /usr/local/bin/nnUNetv2_train  (var=True)


In [11]:
import torch

# ── GPU kontrolü ──────────────────────────────────────────────────────────
_has_cuda = torch.cuda.is_available()
_device   = "cuda" if _has_cuda else "cpu"
print(f"Cihaz: {_device}")

Cihaz: cuda


In [12]:

if _has_cuda:
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

if not _has_cuda:
    print("\n⚠️  CUDA bulunamadı — eğitim GPU gerektirir.")
    print("Bu notebook Colab'da GPU runtime ile çalıştırılmalıdır:")
    print("  Runtime → Change runtime type → T4 GPU (veya A100)")
    raise SystemExit("GPU olmadan eğitim başlatılmadı.")

GPU   : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM  : 102.0 GB


In [13]:
import subprocess
import os

_cmd = _nnunet_cmd("nnUNetv2_train")
# --npz: softmax olasılık haritalarını kaydeder (güven skoru için)
r = subprocess.run(
    [_cmd, str(DATASET_ID), "3d_fullres", "0", "--npz"],
    env={**os.environ}, capture_output=True, text=True, timeout=3600*10
)

print("--- stdout ---")
print(r.stdout)
print("--- stderr ---")
print(r.stderr)

print("Eğitim çıktı kodu:", r.returncode)
if r.returncode != 0:
    print("⚠️  Eğitim hata kodu döndürüyor (bkz. yukarı)")

--- stdout ---

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-06-03 21:27:41.798996: Using torch.compile...
2026-06-03 21:27:42.415161: do_dummy_2d_data_aug: False
2026-06-03 21:27:42.415666: Using splits from existing split file: /content/nnunet_preprocessed/Dataset100_Abdomen/splits_final.json
2026-06-03 21:27:42.415798: The split fil

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 7. 3B Çıkarım

In [14]:
val_cases   = load_fold(fold, "val")

NNUNET_RESULTS =  LOCAL_RESULT
NND_ROOT = 

SyntaxError: invalid syntax (1880857347.py, line 4)

In [ ]:
import shutil, sysconfig, os

# Validasyon seti görüntüleri ayrı predict input dizinine (nnunet_raw DıŞINDA)
# nnU-Net Dataset ID çakışmasından kaçınmak için nnunet_raw dışında tut
VAL_INPUT_DIR  = NND_ROOT / "val_input_raw"
VAL_OUTPUT_DIR = NNUNET_RESULTS / f"Dataset{DATASET_ID}_{DATASET_NAME}" / "nnUNetTrainer__nnUNetPlans__3d_fullres" / "fold_0" / "val_predictions"
VAL_INPUT_DIR.mkdir(parents=True, exist_ok=True)
VAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Val görüntülerini input dizinine semlink (nnunet_raw DIŞINDAKİ dizine)
_linked = 0
for _c in val_cases:
    _src = DATASET_DIR / "imagesTr" / f"case_{_c:05d}_0000.nii.gz"
    _dst = VAL_INPUT_DIR / f"case_{_c:05d}_0000.nii.gz"
    if _src.exists() and not _dst.exists():
        try:
            os.symlink(_src, _dst)
            _linked += 1
        except:
            shutil.copy2(_src, _dst)
            _linked += 1

print(f"Val input dizini (nnunet_raw dışında): {VAL_INPUT_DIR}")
print(f"  → {len(list(VAL_INPUT_DIR.glob('*.nii.gz')))} dosya ({_linked} yeni link/kopya)")

# nnUNetv2_predict
_cmd = _nnunet_cmd("nnUNetv2_predict")
print(f"\nTahmin başlatılıyor: {_cmd}")
r = subprocess.run([
    _cmd,
    "-i", str(VAL_INPUT_DIR),
    "-o", str(VAL_OUTPUT_DIR),
    "-d", str(DATASET_ID),
    "-c", "3d_fullres",
    "-f", "0",
    "--save_probabilities",   # .npz olasılık haritaları (güven skoru için)
], env={**os.environ}, capture_output=True, text=True, timeout=3600*10)

print("STDOUT:\n", r.stdout[-2000:])
if r.returncode != 0:
    print("STDERR:\n", r.stderr[-1000:])
else:
    _preds = list(VAL_OUTPUT_DIR.glob("*.nii.gz"))
    print(f"✓ Tahmin tamamlandı: {len(_preds)} NIfTI mask")

In [ ]:
import numpy as np
import SimpleITK as sitk
from scipy import ndimage
from PIL import Image

PRED_DIR = VAL_OUTPUT_DIR
print("Tahmin dizini:", PRED_DIR)
print(f"→ {len(list(PRED_DIR.glob('*.nii.gz')))} NIfTI mask")

def seg_to_bboxes_2d(pred_nii_path: Path) -> list:
    """
    nnU-Net v2 semantic segmentation mask → 2D-projeksiyon bounding box satırları.

    Her bağlı bileşen (connected component) bir lezyon örneği = bir 3B bbox.
    3B bbox kapsadığı her z-kesitine yansıtılır (nnDetection ile aynı projeksiyon).
    Güven skoru: bileşenin voksel hacmi / tüm o sınıf voksel sayısı.
    .npz olasılık dosyası varsa max softmax olasılığı kullanılır.
    """
    # Case numarası: case_20001.nii.gz → 20001
    try:
        case = int(pred_nii_path.stem.split("_")[1])
    except (IndexError, ValueError):
        return []

    mask = sitk.GetArrayFromImage(sitk.ReadImage(str(pred_nii_path)))  # [Z,Y,X]

    # Olasılık haritası varsa yükle (softmax)
    prob_path = pred_nii_path.with_suffix("").with_suffix(".npz")
    probs = None
    if prob_path.exists():
        probs = np.load(prob_path)["probabilities"]  # [C, Z, Y, X]

    rows = []
    for label_id in range(1, len(SUPER_CLASSES) + 1):
        binary = (mask == label_id).astype(np.uint8)
        if binary.sum() == 0:
            continue
        labeled, n_comp = ndimage.label(binary)
        total_vox = float(binary.sum())

        for comp_id in range(1, n_comp + 1):
            comp_mask = (labeled == comp_id)
            coords = np.where(comp_mask)
            z1, z2 = int(coords[0].min()), int(coords[0].max())
            y1, y2 = int(coords[1].min()), int(coords[1].max())
            x1, x2 = int(coords[2].min()), int(coords[2].max())

            # Güven skoru: softmax prob > hacim bazlı
            if probs is not None:
                score = float(probs[label_id][comp_mask].mean())
            else:
                score = float(comp_mask.sum()) / total_vox

            # 3B bbox'ı kapsadığı her z-kesitine yansıt
            for z in range(z1, z2 + 1):
                rows.append({
                    "case": case, "image_id": z,
                    "class": label_id - 1,   # 0-indexed sınıf
                    "score": score,
                    "x1": float(x1), "y1": float(y1),
                    "x2": float(x2), "y2": float(y2),
                })
    return rows

all_rows = []
for _p in sorted(PRED_DIR.glob("*.nii.gz")):
    all_rows.extend(seg_to_bboxes_2d(_p))

pred_df = pd.DataFrame(all_rows)
print(f"\n3B→2D projeksiyon: {len(pred_df):,} satır, "
      f"{pred_df['case'].nunique() if not pred_df.empty else 0} vaka")
if not pred_df.empty:
    print(pred_df.groupby("class")["score"].describe().round(3))


In [ ]:
from src.evaluation import top5_f1_mean, f1_at_iou

val_lbl_dir = DET_DATA_DIR / f"fold{fold}" / "labels" / "val"
val_img_dir = DET_DATA_DIR / f"fold{fold}" / "images" / "val"

gt_rows = []
for lp in val_lbl_dir.glob("*.txt"):
    ip = val_img_dir / (lp.stem + ".png")
    if not ip.exists():
        continue
    W, H = Image.open(ip).size
    stem = lp.stem
    case, image_id = (stem.split("_", 1) + ["0"])[:2]
    for line in lp.read_text().strip().splitlines():
        p = line.split()
        if len(p) < 5:
            continue
        cl = int(p[0]); cx, cy, w, h = map(float, p[1:5])
        gt_rows.append({
            "case": int(case), "image_id": int(image_id), "class": cl,
            "x1": (cx - w / 2) * W, "y1": (cy - h / 2) * H,
            "x2": (cx + w / 2) * W, "y2": (cy + h / 2) * H,
        })

gt_df = pd.DataFrame(gt_rows)
print(f"GT bbox sayısı: {len(gt_df):,}")

if not pred_df.empty:
    result = top5_f1_mean(pred_df, gt_df)
    detail = f1_at_iou(pred_df, gt_df, iou_th=0.3)
    print(f"\nnnUnet fold {fold} — Top-5 F1 : {result['top5_mean_f1']:.4f}")
    print(f"Macro F1 @ IoU 0.3                : {detail['macro_f1']:.4f}")
    print(f"Micro F1 @ IoU 0.3                : {detail['micro_f1']:.4f}")
else:
    print("Tahmin yok — nndet_predict + nndet_consolidate adımlarının bitmesini bekleyin.")

## 8. Faz 4 Çıktı Özeti

| Çıktı | Yol |
|---|---|
| NIfTI hacimler | `nndet/nifti/case_*.nii.gz` |
| nnU-Net v2 ham veri | `nndet/nnunet_raw/Dataset100_Abdomen/` |
| Preprocessed | `nndet/nnunet_preprocessed/Dataset100_Abdomen/` |
| Eğitilmiş model | `nndet/nnunet_results/Dataset100_Abdomen/nnUNetTrainer__nnUNetPlans__3d_fullres/fold_0/` |
| Tahmin maskeleri | `.../fold_0/val_predictions/*.nii.gz` |
| Olasılık haritaları | `.../fold_0/val_predictions/*.npz` |

**Seg → bbox pipeline:**
Predicted NIfTI mask → her sınıf için binary mask → connected components → 3B bbox → z-ekseni projeksiyon → 2D F1@IoU

**Sonraki:** `Faz5_Harici_Test_Yarisma.ipynb`